# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [52]:
# Load the libraries as required.
%load_ext dotenv
%dotenv 

import os
import sys
import pandas as pd
import numpy as np

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [53]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [54]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, cross_val_score , GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [55]:
X = fires_dt.drop("area", axis=1)
Y = fires_dt['area']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [56]:
# determine and assign the numeric columns
num_cols = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain' ]

#categorical features and one hot encoding
cat_features = ["month", "day"]
#num_transformer_1 = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown="ignore")

pipe_num_simple = Pipeline([
    
    ('standardizer', StandardScaler()),
    #('encoder' , OneHotEncoder(handle_unknown="ignore"))
])

preproc1= ColumnTransformer([
    ('numeric_simple', pipe_num_simple, num_cols),
    ('onehot', cat_transformer, cat_features  )
], remainder='passthrough')

In [57]:
preproc1

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_simple',
                                 Pipeline(steps=[('standardizer',
                                                  StandardScaler())]),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [58]:
pipe_num_simple2 = Pipeline([
    
    ('standardizer', StandardScaler()),
    ('transform', PowerTransformer(method='yeo-johnson'))
])

preproc2 = ColumnTransformer([
    ('numeric_simple', pipe_num_simple2, num_cols),
    ("cat", cat_transformer, cat_features)
], remainder='passthrough')

In [59]:
preproc2

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_simple',
                                 Pipeline(steps=[('standardizer',
                                                  StandardScaler()),
                                                 ('transform',
                                                  PowerTransformer())]),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day'])])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [60]:
# Define Regressors
baseline_regressor = LinearRegression()
advanced_regressor = RandomForestRegressor(random_state=42)

In [61]:
# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", baseline_regressor)
])


In [62]:
pipeline_A

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_simple',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', LinearRegression())])

In [63]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", baseline_regressor)
])


In [64]:
pipeline_B

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_simple',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', LinearRegression())])

In [65]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", advanced_regressor)
])


In [66]:
pipeline_C

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_simple',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [67]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", advanced_regressor)
])

    

In [68]:
pipeline_D

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_simple',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [69]:
# GridSearchCV for tuning (for advanced model)
param_grid_rf = {
    "regressor__n_estimators": [50, 100, 200],
    "regressor__max_depth": [None, 10, 20],
    "regressor__min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(pipeline_C, param_grid_rf, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, Y_train)

# Evaluate with Cross-Validation
pipelines = {"Pipeline A": pipeline_A, "Pipeline B": pipeline_B, "Pipeline C": pipeline_C, "Pipeline D": pipeline_D}

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, Y_train, cv=5, scoring="neg_mean_squared_error")
    print(f"{name}: Mean CV MSE = {-np.mean(scores):.4f}")

# Best Parameters for Advanced Model
print(f"Best Params for RandomForestRegressor: {grid_search.best_params_}")

Pipeline A: Mean CV MSE = 2230.7245
Pipeline B: Mean CV MSE = 2162.0466
Pipeline C: Mean CV MSE = 2783.9712
Pipeline D: Mean CV MSE = 2719.1078
Best Params for RandomForestRegressor: {'regressor__max_depth': 10, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [70]:
# Define parameter grids
param_grid_A = {"regressor__fit_intercept": [True, False]}
param_grid_B = {"regressor__fit_intercept": [True, False]}
param_grid_C = {"regressor__n_estimators": [50, 100, 200, 300]}
param_grid_D = {"regressor__n_estimators": [50, 100, 200, 300]}

# GridSearchCV for each pipeline
grids = {
    "Pipeline A": GridSearchCV(pipeline_A, param_grid_A, cv=5, scoring="neg_mean_squared_error", n_jobs=-1),
    "Pipeline B": GridSearchCV(pipeline_B, param_grid_B, cv=5, scoring="neg_mean_squared_error", n_jobs=-1),
    "Pipeline C": GridSearchCV(pipeline_C, param_grid_C, cv=5, scoring="neg_mean_squared_error", n_jobs=-1),
    "Pipeline D": GridSearchCV(pipeline_D, param_grid_D, cv=5, scoring="neg_mean_squared_error", n_jobs=-1),
}

# Fit GridSearch
best_params = {}
best_scores = {}

for name, grid in grids.items():
    grid.fit(X_train, Y_train)
    best_params[name] = grid.best_params_
    best_scores[name] = -grid.best_score_  # Convert negative MSE to positive MSE
    print(f"{name}: Best Params = {grid.best_params_}, Best MSE = {best_scores[name]:.4f}")

Pipeline A: Best Params = {'regressor__fit_intercept': False}, Best MSE = 2230.3742
Pipeline B: Best Params = {'regressor__fit_intercept': True}, Best MSE = 2162.0466
Pipeline C: Best Params = {'regressor__n_estimators': 100}, Best MSE = 2783.9712
Pipeline D: Best Params = {'regressor__n_estimators': 100}, Best MSE = 2719.1078


# Evaluate

+ Which model has the best performance?

From the results, Pipeline B has the lowest MSE 2162.0466, So it is the best-performing model.

# Export

+ Save the best performing model to a pickle file.

In [71]:
import joblib

# Best-performing model (Pipeline B)
best_model = grids["Pipeline B"].best_estimator_

# Save the model
joblib.dump(best_model, "best_model_pipelineB.pkl")

print("Best model (Pipeline B) saved as 'best_model_pipelineB.pkl'")

Best model (Pipeline B) saved as 'best_model_pipelineB.pkl'


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [72]:
#Use SHAP values to explain the following only for the best-performing model:
#Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.
#In general, across the complete training set, which features are the most and least important.
import shap
# Load the best model
best_model = joblib.load("best_model_pipelineB.pkl")
# Create a SHAP explainer
explainer = shap.Explainer(best_model.named_steps['regressor'], best_model.named_steps['preprocessing'].transform(X_train))
# Select a single observation from the test set
observation_index = 0  # Change this index to select a different observation
observation = X_test.iloc[observation_index:observation_index + 1] 
# Transform the observation using the preprocessing pipeline
observation_transformed = best_model.named_steps['preprocessing'].transform(observation)
# Calculate SHAP values for the observation
shap_values = explainer(observation_transformed)





In [73]:
# To get the feature names I use the below code to populate feature names

# Step 1: Get the fitted preprocessing step from the full pipeline
preprocessor = best_model.named_steps['preprocessing']

# Step 2: Extract feature names after preprocessing
feature_names = []

for name, transformer, cols in preprocessor.transformers_:
    if name == 'remainder' and transformer == 'passthrough':
        # Add the passthrough column names directly
        feature_names.extend(cols)
    else:
        # If the transformer is itself a pipeline (e.g. scaling + imputation), get the final step
        if isinstance(transformer, Pipeline):
            last_step = transformer.steps[-1][1]
        else:
            last_step = transformer

        # Check if the transformer has get_feature_names_out
        if hasattr(last_step, 'get_feature_names_out'):
            # Some transformers need input features to generate names
            try:
                transformed_names = last_step.get_feature_names_out(cols)
            except:
                transformed_names = last_step.get_feature_names_out()
        else:
            # Fallback to raw column names
            transformed_names = cols
        feature_names.extend(transformed_names)

print("Feature names after preprocessing:")
print(feature_names)

Feature names after preprocessing:
['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_fri', 'day_mon', 'day_sat', 'day_sun', 'day_thu', 'day_tue', 'day_wed']


In [74]:
#print the featurs and their imporantance based on SHAP values for this observation
print("SHAP values for the selected observation:")
shap_values_df = pd.DataFrame(shap_values.values, columns=feature_names)
#print the feature names in sorting order based on their importance
shap_values_df = shap_values_df.abs().mean().sort_values(ascending=False)
shap_values_df

SHAP values for the selected observation:


month_may    27.372535
dc           24.151211
dmc          21.586297
temp          5.422280
coord_x       3.911774
month_sep     3.887313
isi           2.592996
month_aug     2.072979
day_sat       1.809293
wind          1.782318
month_mar     1.605869
day_fri       1.132950
ffmc          1.090163
day_sun       0.736618
rain          0.679094
day_thu       0.677407
rh            0.638856
month_jul     0.626881
month_oct     0.582807
month_jun     0.525274
month_apr     0.327643
coord_y       0.313431
month_dec     0.291587
day_tue       0.178061
month_jan     0.105411
day_wed       0.089681
day_mon       0.016826
month_feb     0.015798
month_nov     0.000000
dtype: float64

In [75]:
# Calculate SHAP values for the entire training set
shap_values_train = explainer(best_model.named_steps['preprocessing'].transform(X_train))
#print the featurs and their imporantance based on SHAP values for this observation
print("SHAP values for the entire trainingset:")
shap_values_train_df = pd.DataFrame(shap_values_train.values, columns=feature_names)
#print the feature names in sorting order based on their importance
shap_values_train_df = shap_values_train_df.abs().mean().sort_values(ascending=False)
shap_values_train_df

SHAP values for the entire trainingset:


dc           14.522135
dmc          12.752994
month_sep     5.264373
coord_x       4.691787
temp          3.741373
month_aug     3.009919
month_mar     2.445743
wind          2.146355
day_fri       1.519287
month_jul     1.419209
day_thu       1.379782
isi           1.373613
day_sun       1.320293
month_jun     1.318909
month_oct     1.157617
rain          0.906006
ffmc          0.566057
day_sat       0.561007
month_apr     0.501645
month_dec     0.424319
coord_y       0.393000
day_tue       0.282685
rh            0.233814
day_wed       0.170796
month_jan     0.130423
month_may     0.066277
day_mon       0.033339
month_nov     0.026634
month_feb     0.025897
dtype: float64

*(Answer here.)*

+ In general, across the complete training set, which features are the most and least important.
    - As seen above, the most important feature is "dc" and least important feature is "month_feb"

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?
    - features with low SHAP values shall be removed because it's importance in the prediction process is less. In order test that, those features have to be removed from the training data and the test data and compare the improvement of the model.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.